# 01 数据检查

本 Notebook 用于阶段 1 的只读数据检查，不修改 `data/raw/`。

## 代码说明：导入依赖

这段代码解决环境准备问题。需要先导入路径、pandas 和项目读取函数，才能自动定位并读取附件。

In [ ]:
from pathlib import Path
import pandas as pd
from src.data_loader import find_raw_file, read_excel_workbook, SALES_KEYWORDS, WEATHER_KEYWORDS
from src.preprocessing import summarize_frame, standardize_sales, standardize_weather, one_to_one_violations


输出怎么看：如果导入没有报错，说明项目模块可以被 Notebook 使用。若报错，优先检查工作目录是否为项目根目录。

## 代码说明：读取原始工作簿

这段代码解决“附件在哪里、有哪些 sheet”的问题。阶段 1 必须逐个 sheet 读取，避免误把空表当成有效数据。

In [ ]:
sales_path = find_raw_file(SALES_KEYWORDS)
weather_path = find_raw_file(WEATHER_KEYWORDS)
sales_sheets = read_excel_workbook(sales_path)
weather_sheets = read_excel_workbook(weather_path)
print(sales_path)
print(weather_path)
print(sales_sheets.keys())
print(weather_sheets.keys())


输出怎么看：应看到附件一和附件二的真实文件路径，以及附件二包含 `Sheet1`、`Sheet2`、`Sheet3`。空表不参与建模，但要在报告中说明。

## 代码说明：表结构检查

这段代码输出每张表的行列数、字段类型、缺失值和前 5 行。这样处理是为了先看清数据结构，再决定清洗规则。

In [ ]:
for label, sheets in [('附件一', sales_sheets), ('附件二', weather_sheets)]:
    for sheet_name, df in sheets.items():
        print('\n', label, sheet_name)
        print('shape:', df.shape)
        print('dtypes:')
        print(df.dtypes)
        print('missing:')
        print(df.isna().sum())
        print('duplicates:', df.duplicated().sum())
        display(df.head())


输出怎么看：附件一应是销售明细表；附件二 `Sheet1` 是有效外部变量表，其他 sheet 为空。若字段名变化，后续标准化代码需要同步修改。

## 代码说明：质量检查

这段代码检查日期范围、门店商品类别数量、负销量、编号名称映射关系。这样做是为了找出会影响建模口径的问题。

In [ ]:
raw_sales = next(df for df in sales_sheets.values() if df.shape[0] > 0 or df.shape[1] > 0)
raw_weather = next(df for df in weather_sheets.values() if df.shape[0] > 0 or df.shape[1] > 0)
sales = standardize_sales(raw_sales)
weather = standardize_weather(raw_weather)
print('销售日期范围:', sales['date'].min(), sales['date'].max())
print('天气日期范围:', weather['date'].min(), weather['date'].max())
print('门店数量:', sales['store_id'].nunique())
print('商品代码数量:', sales['product_id'].nunique())
print('商品名称数量:', sales['product_name'].nunique())
print('类别数量:', sales['category'].nunique())
print('负销量记录数:', (sales['quantity'] < 0).sum())
print('商品代码到商品名称异常:')
display(one_to_one_violations(sales, 'product_id', 'product_name'))


输出怎么看：如果出现负销量或商品编号名称不一一对应，后续不能直接删除或合并，必须在预处理表中保留标记并在论文中说明。